In [1]:
import duckdb

In [2]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [3]:
df = con.execute('''
        select * from(
            select *, row_number() over (partition by NATBR order by data_ingestao desc) as rn 
            from bronze_z0019
            where data_ingestao >= '2026-05-04'
        ) where rn = 1
    ''').fetchdf()

df.head(10)


,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,rn
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-05-04 16:42:40.500904,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-05-04 16:42:40.500904,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-05-04 16:56:45.727173,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-05-04 16:56:45.727173,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-05-04 16:56:45.727173,1


In [6]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'rn'])
df_final.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [7]:
df_final = df_final.rename(columns={'NATBR': 'id', 'MAKTX': 'nm_produto', 'WERKS': 'id_categoria', 'MAINS': 'id_fornecedor', 'LABST': 'vl_preco'})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [18]:
df2.dtypes

id                        int32
nm_produto       string[python]
id_categoria     string[python]
id_fornecedor             int32
vl_preco                float32
dtype: object

In [21]:
df2 = df_final
df2 = df2.astype(
    {
        'id': 'int32', 
        'nm_produto': 'string',
        'id_categoria': 'string',
        'id_fornecedor': 'int32',
        'vl_preco': 'float32'
    }
)
df2.dtypes

id                        int32
nm_produto       string[python]
id_categoria     string[python]
id_fornecedor             int32
vl_preco                float32
dtype: object

In [22]:
con.execute('''
    CREATE TABLE IF NOT EXISTS produtos (
        id INTEGER,
        nm_produto STRING,
        id_categoria STRING,
        id_fornecedor INTEGER,
        vl_preco FLOAT
    )
''')

In [23]:
con.execute('''insert into produtos select * from df2''')

In [24]:
df_resultado = con.execute('select * from produtos').fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [25]:
con.close()